# RadGraph-XL Data Audit

This notebook reads the credentialed RadGraph-XL ZIP from the local path in `.env`, verifies the JSONL structure, and writes aggregate statistics only. It does not display or export report text.

In [ ]:
from __future__ import annotations

import csv
import hashlib
import json
import math
import os
import statistics
import zipfile
from collections import Counter
from pathlib import Path

from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / ".env").exists() and (PROJECT_ROOT.parent / ".env").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
load_dotenv(PROJECT_ROOT / ".env")

zip_path = Path(os.environ["RADGRAPH_XL_ZIP"])
assert zip_path.exists(), f"RadGraph-XL ZIP not found: {zip_path}"

audit_dir = PROJECT_ROOT / "outputs" / "audit"
audit_dir.mkdir(parents=True, exist_ok=True)

print(zip_path)

In [ ]:
with zipfile.ZipFile(zip_path) as archive:
    jsonl_members = [name for name in archive.namelist() if name.lower().endswith(".jsonl")]

assert len(jsonl_members) == 1, jsonl_members
jsonl_member = jsonl_members[0]

sha256 = hashlib.sha256()
records = []
with zipfile.ZipFile(zip_path) as archive:
    with archive.open(jsonl_member) as handle:
        for line in handle:
            sha256.update(line)
            if line.strip():
                records.append(json.loads(line))

print({"jsonl_member": jsonl_member, "records": len(records), "jsonl_sha256": sha256.hexdigest()})

In [ ]:
def flatten_entities(record):
    return [entity for sentence_entities in record["ner"] for entity in sentence_entities]


def flatten_relations(record):
    return [relation for sentence_relations in record["relations"] for relation in sentence_relations]


def token_count(record):
    return sum(len(sentence) for sentence in record["sentences"])


def relation_gap(relation):
    source_start, source_end, target_start, target_end, _label = relation
    return max(0, max(target_start - source_end, source_start - target_end))


def distribution(values):
    if not values:
        return {"min": None, "median": None, "mean": None, "p90": None, "p95": None, "max": None}
    ordered = sorted(values)

    def percentile(p):
        index = min(len(ordered) - 1, max(0, math.ceil(p * len(ordered)) - 1))
        return ordered[index]

    return {
        "min": min(values),
        "median": statistics.median(values),
        "mean": round(statistics.mean(values), 2),
        "p90": percentile(0.90),
        "p95": percentile(0.95),
        "max": max(values),
    }

In [ ]:
datasets = Counter(record["dataset"] for record in records)
token_counts = [token_count(record) for record in records]
entity_counts = [len(flatten_entities(record)) for record in records]
relation_counts = [len(flatten_relations(record)) for record in records]

entity_labels = Counter(entity[2] for record in records for entity in flatten_entities(record))
entity_categories = Counter(label.split("::", maxsplit=1)[0] for label in entity_labels.elements())
entity_assertions = Counter(label.split("::", maxsplit=1)[1] for label in entity_labels.elements())
entity_span_lengths = [entity[1] - entity[0] + 1 for record in records for entity in flatten_entities(record)]

relation_labels = Counter(relation[4] for record in records for relation in flatten_relations(record))
relation_gaps = [relation_gap(relation) for record in records for relation in flatten_relations(record)]
distance_thresholds = [8, 16, 24, 32, 48, 64, 80, 96, 128]

summary = {
    "source": {
        "zip_path": str(zip_path),
        "zip_size_bytes": zip_path.stat().st_size,
        "jsonl_member": jsonl_member,
        "jsonl_sha256": sha256.hexdigest(),
    },
    "documents": {
        "total": len(records),
        "datasets": dict(sorted(datasets.items())),
    },
    "tokens": {
        **distribution(token_counts),
        "total": sum(token_counts),
        "reports_gt_384": sum(count > 384 for count in token_counts),
        "reports_gt_512": sum(count > 512 for count in token_counts),
    },
    "entities": {
        "total": sum(entity_counts),
        "per_report": distribution(entity_counts),
        "labels": dict(entity_labels.most_common()),
        "categories": dict(entity_categories.most_common()),
        "assertions": dict(entity_assertions.most_common()),
        "span_lengths": distribution(entity_span_lengths),
    },
    "relations": {
        "total": sum(relation_counts),
        "per_report": distribution(relation_counts),
        "labels": dict(relation_labels.most_common()),
        "distance_gaps": distribution(relation_gaps),
        "distance_threshold_coverage": {
            str(threshold): {
                "covered": sum(gap <= threshold for gap in relation_gaps),
                "coverage": round(sum(gap <= threshold for gap in relation_gaps) / len(relation_gaps), 6),
            }
            for threshold in distance_thresholds
        },
    },
}

print(json.dumps({
    "reports": summary["documents"]["total"],
    "entities": summary["entities"]["total"],
    "relations": summary["relations"]["total"],
    "datasets": summary["documents"]["datasets"],
}, indent=2))

In [ ]:
audit_json = audit_dir / "data_audit.json"
report_csv = audit_dir / "report_summary.csv"

audit_json.write_text(json.dumps(summary, indent=2), encoding="utf-8")

with report_csv.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=["doc_key", "dataset", "sentence_count", "token_count", "entity_count", "relation_count"],
    )
    writer.writeheader()
    for record in records:
        writer.writerow({
            "doc_key": record["doc_key"],
            "dataset": record["dataset"],
            "sentence_count": len(record["sentences"]),
            "token_count": token_count(record),
            "entity_count": len(flatten_entities(record)),
            "relation_count": len(flatten_relations(record)),
        })

print({"audit_json": str(audit_json), "report_csv": str(report_csv)})